In [1]:
%cd ..

/Users/ext-elias.melas/Documents/Gitcode/opensearch-testing


In [4]:
from opensearchpy import OpenSearch
from src.utils import search_utils
from dotenv import load_dotenv
load_dotenv()

# client = OpenSearch(
#     hosts=[{'host': 'localhost', 'port': 4434, 'scheme': 'https'}],  # Explicitly use HTTP
#     http_auth=('admin', 'admin'),  # Default credentials for GitHub Actions OpenSearch
#     use_ssl=False,
#     verify_certs=False,
#     ssl_show_warn=False,
# )

from src.utils.search_utils import GameAccountSearcher, QueryType

from importlib import reload

# Create Mock data (100k accounts)

In [3]:
# from mock_data.make_index_mock import main
# main()

## Search Mock data

In [3]:
# Basic search
# searcher = GameAccountSearcher(client)
# results = searcher.add_query("player_tag", "ABC12", QueryType.TERM).search()

# Complex search with multiple condition
searcher = GameAccountSearcher(test=False,index='soil_accounts')
results = (
    searcher
    # .add_query('name', 'zacharywallace', QueryType.MATCH)
    # .add_query('name', 'melas', QueryType.MATCH)
    .add_query("name", "emelas", QueryType.TERM,)
    .add_query("exp_level", "7", QueryType.TERM,)
    # .add_query("alliance_name", "dragon war", QueryType.MATCH, fuzziness="AUTO")
    # .add_query(field='create_country', value='uk', query_type=QueryType.TERM)
    .search()
)

results


testing mode: False
client initialised: <OpenSearch([{'host': 'localhost', 'port': 4434}])>
index: soil_accounts
body: {'query': {'bool': {'must': [{'term': {'name': 'emelas'}}, {'term': {'exp_level': '7'}}]}}, 'size': 10}
client: <OpenSearch([{'host': 'localhost', 'port': 4434}])>


{'took': 97,
 'timed_out': False,
 '_shards': {'total': 16, 'successful': 16, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 1, 'relation': 'eq'},
  'max_score': 16.20082,
  'hits': [{'_index': 'soil_accounts',
    '_type': '_doc',
    '_id': '25-41476889',
    '_score': 16.20082,
    '_source': {'exp_level': 7,
     'account_id': 107415659289,
     'avatar_id': 107415659289,
     'updated': 1749035769469,
     'create_country': 'GB',
     'name': 'emelas farm',
     'name_raw': 'emelas farm',
     'name_raw_lower': 'emelas farm',
     'name_raw_rev': 'mraf saleme',
     'name_raw_lower_rev': 'mraf saleme',
     'alliance_name': '',
     'alliance_name_raw': '',
     'alliance_name_raw_lower': ''}}]},
 'no_of_hits': 1,
 'found': False,
 'results': [{'exp_level': 7,
   'account_id': 107415659289,
   'avatar_id': 107415659289,
   'updated': 1749035769469,
   'create_country': 'GB',
   'name': 'emelas farm',
   'name_raw': 'emelas farm',
   'name_raw_lower': 'emelas farm',
   'n

In [ ]:
# Multi-field search
searcher = GameAccountSearcher(client)
results = searcher.add_query(
    "search_term",
    "dragon",
    QueryType.MULTI_MATCH,
    fields=["alliance_name_raw",]
).search()

results

## Structured Output

### Single Message

In [24]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional

client = OpenAI()

class AccountParams(BaseModel):
    name: Optional[str] = Field(default=None, description="The name of the account")
    alliance_name: Optional[str] = Field(default=None, description="The name of the alliance")
    updated : Optional[str] = Field(default=None, description="The last updated date of the account")
    create_country: Optional[str] = Field(default=None, description="The country of the account")
    exp_level: Optional[str] = Field(default=None, description="The experience level of the account")
    updated: Optional[str] = Field(default=None, description="The last updated date of the account")
    

field_rankings = {
    'name': 1,
    'alliance_name': 2,
    'updated': 3,
    'create_country': 4,
    'exp_level': 5
}

system_prompt = f"""
You are a game account search assistant that helps extract account information from a user conversation with a game agent.
The rankings for the fields are as follows:
{field_rankings}

"""

system_prompt = """
You are a game account search assistant helping a user find their missing account.
Your goal is to gather information systematically based on field importance rankings.

Field Rankings (1=most important, 5=least important):
{
  "name": 1,
  "alliance_name": 2,
  "updated": 3,
  "create_country": 4,
  "exp_level": 5
}

Current iteration: 1/3


Instructions:
1. Extract any account information from the user's message.
2. If one of the fields seem vague or generic, don't be afraid to prompt again for the same information.
3. If results are decreasing significantly, ask more specific clarifying questions
4. Focus on the highest-ranked fields first (name is most important)
5. Ask one focused question at a time to refine the search
6. Be conversational and helpful

Current known information:
None collected yet

Please respond with a JSON object containing only the extracted parameters. Use null for any fields where no information was found.
"""

user_query = "I want to find an account with the name zacharywallace, from uk"

response = client.responses.parse(
    model="gpt-4.1-mini",
    input=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": user_query,
        },
    ],
    text_format=AccountParams,
)

account_params = response.output_parsed

In [25]:
account_params.__dict__

{'name': 'zacharywallace',
 'alliance_name': None,
 'updated': None,
 'create_country': 'uk',
 'exp_level': None}

### Multiple questions

In [28]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional, Dict, Any, List, Tuple
from enum import Enum
import json

from src.utils.search_utils import GameAccountSearcher, QueryType

client = OpenAI()

class AccountParams(BaseModel):
    name: Optional[str] = Field(default=None, description="The name of the account")
    alliance_name: Optional[str] = Field(default=None, description="The name of the alliance")
    updated: Optional[str] = Field(default=None, description="The last updated date of the account")
    create_country: Optional[str] = Field(default=None, description="The country of the account")
    exp_level: Optional[str] = Field(default=None, description="The experience level of the account")

class AccountRecoveryAgent:
    def __init__(self, searcher, min_threshold: int = 100, max_iterations: int = 3):
        self.searcher = searcher
        self.min_threshold = min_threshold
        self.max_iterations = max_iterations
        self.field_rankings = {
            'name': 1,
            'alliance_name': 2,
            'updated': 3,
            'create_country': 4,
            'exp_level': 5
        }
        self.conversation_history = []
        self.search_history = []
        self.current_params = AccountParams()
        
    def get_system_prompt(self, iteration: int, last_hits: int = None, trend: str = None) -> str:
        trend_info = ""
        if last_hits is not None:
            trend_info = f"\nLast search returned {last_hits} results."
            if trend:
                trend_info += f" Results are {trend}."
        
        return f"""
You are a game account search assistant helping a user find their missing account.
Your goal is to gather information systematically based on field importance rankings.

Field Rankings (1=most important, 5=least important):
{json.dumps(self.field_rankings, indent=2)}

Current iteration: {iteration + 1}/{self.max_iterations}
{trend_info}

Instructions:
1. Extract any account information from the user's message.
2. If one of the fields seem vague or generic, don't be afraid to prompt again for the same information.
3. If results are decreasing significantly, ask more specific clarifying questions
4. Focus on the highest-ranked fields first (name is most important)
5. Ask one focused question at a time to refine the search
6. Be conversational and helpful

Current known information:
{self._format_current_params()}
"""

    def _format_current_params(self) -> str:
        params_dict = self.current_params.model_dump(exclude_none=True)
        if not params_dict:
            return "None collected yet"
        return json.dumps(params_dict, indent=2)
    
    def extract_params_from_response(self, user_input: str, iteration: int, last_hits: int = None, trend: str = None) -> Tuple[AccountParams, str]:
        """Extract parameters and get agent's next question"""
        system_prompt = self.get_system_prompt(iteration, last_hits, trend)
        
        # Build conversation context
        messages = [{"role": "system", "content": system_prompt}]
        messages.extend(self.conversation_history)
        messages.append({"role": "user", "content": user_input})
        
        # Extract structured data using correct OpenAI API
        param_response = client.responses.parse(
            model="gpt-4.1-mini",
            input=messages,
            text_format=AccountParams,
        )
        
        extracted_params = param_response.output_parsed
        
        # Get conversational response using correct API
        chat_response = client.responses.create(
            model="gpt-4.1-mini",
            input=messages + [
                {"role": "assistant", "content": f"I found this information: {extracted_params.model_dump(exclude_none=True)}"},
                {"role": "user", "content": "What should you ask next to help narrow down the search?"}
            ],
            # max_tokens=200

        )
        
        next_question = chat_response.output_text
        return extracted_params, next_question

    def merge_params(self, new_params: AccountParams) -> None:
        """Merge new parameters with existing ones"""
        current_dict = self.current_params.model_dump()
        new_dict = new_params.model_dump(exclude_none=True)
        
        for key, value in new_dict.items():
            if value is not None:
                current_dict[key] = value
        
        self.current_params = AccountParams(**current_dict)

    def build_search_query(self) -> 'GameAccountSearcher':
        """Build search query based on current parameters - creates fresh searcher instance"""
        # Create a new searcher instance for each search
        query_builder = GameAccountSearcher(self.searcher.client)
        
        params_dict = self.current_params.model_dump(exclude_none=True)
        
        # Sort by field rankings (lower number = higher priority)
        sorted_params = sorted(
            params_dict.items(), 
            key=lambda x: self.field_rankings.get(x[0], 999)
        )
        
        for field, value in sorted_params:
            if field == 'name':
                query_builder = query_builder.add_query(field, value, QueryType.MATCH)
            elif field == 'alliance_name':
                query_builder = query_builder.add_query(field, value, QueryType.MATCH, fuzziness="AUTO")
            else:
                query_builder = query_builder.add_query(field, value, QueryType.MATCH)
        
        return query_builder

    def calculate_trend(self, current_hits: int) -> str:
        """Calculate if results are improving or declining"""
        if len(self.search_history) < 1:
            return "initial"
        
        previous_hits = self.search_history[-1]['hits']
        
        if current_hits < previous_hits * 0.5:  # 50% decrease
            return "decreasing significantly"
        elif current_hits < previous_hits:
            return "decreasing slightly"
        elif current_hits > previous_hits * 1.5:  # 50% increase
            return "improving significantly"
        elif current_hits > previous_hits:
            return "improving slightly"
        else:
            return "stable"

    def run_recovery_session(self, initial_query: str) -> Dict[str, Any]:
        """Main recovery session loop"""
        print("🔍 Starting account recovery session...")
        print(f"Minimum threshold: {self.min_threshold} results")
        print(f"Maximum iterations: {self.max_iterations}")
        print("-" * 50)
        
        current_input = initial_query
        
        for iteration in range(self.max_iterations):
            print(f"\n🔄 Iteration {iteration + 1}")
            
            # Calculate trend from previous searches
            last_hits = self.search_history[-1]['hits'] if self.search_history else None
            trend = self.calculate_trend(last_hits) if last_hits is not None else None
            
            # Extract parameters and get next question
            new_params, next_question = self.extract_params_from_response(
                current_input, iteration, last_hits, trend
            )
            
            # Merge with existing parameters
            self.merge_params(new_params)
            
            print(f"📝 Current parameters: {self.current_params.model_dump(exclude_none=True)}")
            
            # Perform search
            if any(self.current_params.model_dump(exclude_none=True).values()):
                # try:
                query_builder = self.build_search_query()
                results = query_builder.search()
                print(results)
                
                hits = results['no_of_hits']
                current_trend = self.calculate_trend(hits)
                
                self.search_history.append({
                    'iteration': iteration + 1,
                    'params': self.current_params.model_dump(exclude_none=True),
                    'hits': hits,
                    'trend': current_trend
                })
                
                print(f"🎯 Search results: {hits} hits ({current_trend})")
                
                # Check if we should stop

                if hits == 1:
                    print('Account found!')
                    return {
                        'status': 'success',
                        'final_results': results,
                        'iterations': iteration + 1,
                        'final_params': self.current_params.model_dump(exclude_none=True),
                        'search_history': self.search_history
                    }
                elif hits <= self.min_threshold < 0:
                    print(f"✅ Found {hits} results (below threshold of {self.min_threshold})")
                    return {
                        'status': 'success',
                        'final_results': results,
                        'iterations': iteration + 1,
                        'final_params': self.current_params.model_dump(exclude_none=True),
                        'search_history': self.search_history
                    }
                
                # If results are decreasing significantly, modify the question
                if current_trend == "decreasing significantly":
                    next_question = f"⚠️ Results dropped to {hits} (from {self.search_history[-2]['hits'] if len(self.search_history) > 1 else 'unknown'}). {next_question}"
                
                # except Exception as e:
                #     print(f"❌ Search error: {e}")
                #     next_question = f"There was a search error. {next_question}"
            
            # Add to conversation history
            self.conversation_history.extend([
                {"role": "user", "content": current_input},
                {"role": "assistant", "content": next_question}
            ])
            
            print(f"🤖 Agent: {next_question}")
            
            # Get user input for next iteration
            if iteration < self.max_iterations - 1:  # Don't ask on last iteration
                current_input = input(f"\n👤 Your response: ")
                if not current_input.strip():
                    print("Empty input, stopping session.")
                    break
        
        # Max iterations reached
        final_hits = self.search_history[-1]['hits'] if self.search_history else 0
        return {
            'status': 'max_iterations_reached',
            'final_results': None,
            'iterations': self.max_iterations,
            'final_params': self.current_params.model_dump(exclude_none=True),
            'search_history': self.search_history,
            'final_hits': final_hits
        }

# Example usage with proper GameAccountSearcher integration
def example_usage():
    """
    Example of how to use the AccountRecoveryAgent with GameAccountSearcher
    """

    searcher = GameAccountSearcher()
    
    # Initialize the agent
    agent = AccountRecoveryAgent(
        searcher=searcher,
        min_threshold=5,
        max_iterations=3
    )
    
    # Start recovery session
    initial_query = "I want to find an account with the name zacharywallace, from uk"
    
    # Run the session
    result = agent.run_recovery_session(initial_query)
    
    print("\n" + "="*50)
    print("SESSION SUMMARY")
    print("="*50)
    print(f"Status: {result['status']}")
    print(f"Iterations: {result['iterations']}")
    print(f"Final parameters: {result['final_params']}")
    
    print("\nSearch History:")
    for search in result['search_history']:
        print(f"  Iteration {search['iteration']}: {search['hits']} hits ({search['trend']})")


# Your integration:

# Create searcher with your client
searcher = GameAccountSearcher(client)

# Initialize and run agent
agent = AccountRecoveryAgent(searcher=searcher, min_threshold=1, max_iterations=5)
result = agent.run_recovery_session("I want to find an account called zacharywallace. i was in an alliance with the word dragon ")


print(result)

testing mode: True
client initialised: <OpenSearch([{'host': 'localhost', 'port': 9200, 'scheme': 'http'}])>
🔍 Starting account recovery session...
Minimum threshold: 1 results
Maximum iterations: 5
--------------------------------------------------

🔄 Iteration 1


KeyboardInterrupt: 

In [ ]:
result

## Testing Graph Agent

In [ ]:
from src.agents import account_recovery_graph
from pydantic_graph import BaseNode, End, Graph, GraphRunContext
from src.config import base_models
import json

reload(account_recovery_graph)
reload(search_utils)
reload(base_models)

# Initialize state with first query
state =  base_models.State(
    test=True,
    index = 'game_accounts',
    conversation_history=[{"role": "user", "content": ''}],
    min_threshold=5,
    max_iterations=3
)

state = await account_recovery_graph.main(state)


🔍 Account Recovery Assistant
Please provide information about the account you're looking for.
You can include details like:
- Account name
- Alliance name
- Last updated date
- Country
- Experience level


2025-06-04 15:26:13,958 - src.agents.account_recovery_graph - INFO - Starting PrepareMessageHistoryNode execution
2025-06-04 15:26:13,958 - src.agents.account_recovery_graph - INFO - Starting ExtractParamsNode execution
2025-06-04 15:26:13,959 - src.agents.account_recovery_graph - INFO - Messages being sent to API: [
  {
    "role": "system",
    "content": "\nYou are a game account search assistant helping a user find their missing account.\nYour goal is to gather information systematically based on field importance rankings.\n\nField Rankings (1=most important, 5=least important):\n{\n  \"name\": 1,\n  \"alliance_name\": 2,\n  \"updated\": 3,\n  \"create_country\": 4,\n  \"exp_level\": 5\n}\n\nCurrent iteration: 1/3\n\n\nInstructions:\n1. Extract any account information from the user's message.\n2. If one of the fields seem vague or generic, don't be afraid to prompt again for the same information.\n3. If results are decreasing significantly, ask more specific clarifying questions\n4


Processing your request...


2025-06-04 15:26:16,971 - src.agents.account_recovery_graph - INFO - Raw API Response: ParsedResponse[AccountParams](id='resp_684057861a60819bb6be98303c5ab6340433eb9a260b2a7c', created_at=1749047174.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-mini-2025-04-14', object='response', output=[ParsedResponseOutputMessage[AccountParams](id='msg_6840578825c8819b8b3bf8719cac49790433eb9a260b2a7c', content=[ParsedResponseOutputText[AccountParams](annotations=[], text='{"name":null,"alliance_name":null,"updated":null,"create_country":null,"exp_level":null}', type='output_text', parsed=AccountParams(name=None, alliance_name=None, updated=None, create_country=None, exp_level=None))], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, max_output_tokens=None, previous_response_id=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None),

testing mode: True
client initialised: <OpenSearch([{'host': 'localhost', 'port': 9200, 'scheme': 'http'}])>
index: game_accounts
body: {'query': {'bool': {}}, 'size': 10}
client: <OpenSearch([{'host': 'localhost', 'port': 9200, 'scheme': 'http'}])>


2025-06-04 15:26:24,914 - src.agents.account_recovery_graph - INFO - LLM response generated successfully
2025-06-04 15:26:24,915 - src.agents.account_recovery_graph - INFO - Starting UserInputNode execution



ASSISTANT RESPONSE
I found several potential matches for your missing game account. Here are some details to help you identify your account:

1. Name: deborah36
   - Level: 12
   - Alliance: Crystal Phoenix
   - Created in: KP

2. Name: matthewhernandez
   - Level: 12
   - Alliance: Dragon Warriors
   - Created in: SE

3. Name: lopezerica
   - Level: 1
   - Alliance: Silver Wolves
   - Created in: SR

4. Name: marcusshaffer
   - Level: 44
   - Alliance: Emerald Guardians
   - Created in: CM

5. Name: lisaknight
   - Level: 45
   - Alliance: Crimson Dragons
   - Created in: AG

6. Name: robinsonholly
   - Level: 47
   - Alliance: Thunder Legion
   - Created in: NZ

7. Name: njenkins
   - Level: 11
   - Alliance: Silver Wolves
   - Created in: KN

8. Name: michael93
   - Level: 59
   - Alliance: Silver Wolves
   - Created in: SM

9. Name: heidi07
   - Level: 48
   - Alliance: Shadow Knights
   - Created in: BI

10. Name: barnesdeborah
    - Level: 82
    - Alliance: Thunder Legion
    -

2025-06-04 15:26:38,890 - src.agents.account_recovery_graph - INFO - Empty input, stopping session



ASSISTANT RESPONSE
I found several potential matches for your missing game account. Here are some details to help you identify your account:

1. Name: deborah36
   - Level: 12
   - Alliance: Crystal Phoenix
   - Created in: KP

2. Name: matthewhernandez
   - Level: 12
   - Alliance: Dragon Warriors
   - Created in: SE

3. Name: lopezerica
   - Level: 1
   - Alliance: Silver Wolves
   - Created in: SR

4. Name: marcusshaffer
   - Level: 44
   - Alliance: Emerald Guardians
   - Created in: CM

5. Name: lisaknight
   - Level: 45
   - Alliance: Crimson Dragons
   - Created in: AG

6. Name: robinsonholly
   - Level: 47
   - Alliance: Thunder Legion
   - Created in: NZ

7. Name: njenkins
   - Level: 11
   - Alliance: Silver Wolves
   - Created in: KN

8. Name: michael93
   - Level: 59
   - Alliance: Silver Wolves
   - Created in: SM

9. Name: heidi07
   - Level: 48
   - Alliance: Shadow Knights
   - Created in: BI

10. Name: barnesdeborah
    - Level: 82
    - Alliance: Thunder Legion
    -

In [ ]:
state.current_results.__dict__